# STEP-003 — Kaggle `DS0001` Audit, Verification & Master Index CSV

**Project:** AI-Generated Face Detection for Digital Identity Verification (e-KYC) and Fraud Prevention (`GROUP 08`)  
**Owner:** Member 1 — Md Assaduzzaman (`0242310005101233`)  
**Dataset:** `kaggle:isthisdeception/ds0001-artifact-face-subset` (ArtiFact Face Subset, MIT License)  

### Objectives
1. Auto-locate `DS0001` on Kaggle and load its `metadata.csv` label/generator mapping.
2. Scan all 50,000 face images, verifying readability (`Pillow`), dimensions, color mode, format, and computing exact MD5 checksums for zero-leakage splitting (`STEP-005`).
3. Export two lightweight files to `/kaggle/working/` to download and commit to GitHub (`02_data/`):
   - `ds0001_index.csv` (`image_id,rel_path,label,label_name,generator,width,height,file_ext,md5_hash`)
   - `ds0001_audit_summary.json`
4. Render an annotated sample preview (`ds0001_annotation_examples.png`) for Milestone 2 evidence.


In [ ]:
import hashlib
import json
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("02_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR.resolve())


In [ ]:
# 1. Auto-detect DS0001 root directory on Kaggle (supports both classic and datasets/ mount layouts)
CANDIDATE_ROOTS = [
    Path("/kaggle/input/datasets/isthisdeception/ds0001-artifact-face-subset"),
    Path("/kaggle/input/ds0001-artifact-face-subset"),
]

MOUNT_ROOT = None
for cand in CANDIDATE_ROOTS:
    if cand.exists():
        MOUNT_ROOT = cand
        break

if MOUNT_ROOT is None:
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = [p for p in kaggle_input.rglob("*ds0001*") if p.is_dir()]
        if matches:
            MOUNT_ROOT = matches[0]

assert MOUNT_ROOT is not None, (
    "DS0001 not found! Attach Kaggle dataset 'isthisdeception/ds0001-artifact-face-subset' to this notebook."
)

# Locate the actual directory containing real/ and fake/ image folders
DATA_ROOT = MOUNT_ROOT
for p in [MOUNT_ROOT, *MOUNT_ROOT.rglob("*")]:
    if p.is_dir() and (p / "real").is_dir() and (p / "fake").is_dir():
        DATA_ROOT = p
        break

METADATA_CSVS = sorted(MOUNT_ROOT.rglob("metadata.csv"))
print("MOUNT_ROOT   :", MOUNT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)
print("METADATA_CSVS:", [str(p) for p in METADATA_CSVS])


In [ ]:
# 2. Load metadata.csv mapping -> {key: (label_name, generator)}
def norm_path(p: str) -> str:
    return str(p).replace("\\", "/").strip().lstrip("./")

meta_lookup = {}
for csv_path in METADATA_CSVS:
    df_meta = pd.read_csv(csv_path)
    print(f"Loaded {csv_path.name} ({len(df_meta)} rows) | columns: {list(df_meta.columns)}")
    cols = {c.lower(): c for c in df_meta.columns}
    fn_col = cols.get("filename") or cols.get("file_name") or cols.get("image")
    op_col = cols.get("output_path") or cols.get("rel_path") or cols.get("path")
    lbl_col = cols.get("label") or cols.get("class")
    gen_col = cols.get("generator") or cols.get("source") or cols.get("model")

    for row in df_meta.itertuples(index=False):
        row_dict = row._asdict()
        raw_lbl = str(row_dict.get(lbl_col, "")).strip().lower() if lbl_col else ""
        if raw_lbl in ("0", "real", "pristine"):
            label_name = "real"
        elif raw_lbl in ("1", "fake", "ai", "synthetic"):
            label_name = "fake"
        else:
            label_name = raw_lbl or "unknown"

        raw_gen = str(row_dict.get(gen_col, "")).strip().lower() if gen_col else ""
        if label_name == "real" and (not raw_gen or raw_gen in ("nan", "none", "null")):
            generator = "none"
        else:
            generator = raw_gen if raw_gen and raw_gen != "nan" else ("none" if label_name == "real" else "unknown")

        entry = (label_name, generator)
        if fn_col and pd.notna(row_dict.get(fn_col)):
            fn = norm_path(row_dict[fn_col])
            meta_lookup[fn] = entry
            meta_lookup[Path(fn).name] = entry
            meta_lookup[f"real/{Path(fn).name}"] = entry
            meta_lookup[f"fake/{Path(fn).name}"] = entry
        if op_col and pd.notna(row_dict.get(op_col)):
            op = norm_path(row_dict[op_col])
            meta_lookup[op] = entry
            meta_lookup[Path(op).name] = entry
            parts = op.split("/")
            if len(parts) >= 2:
                meta_lookup["/".join(parts[-2:])] = entry

print("Total lookup keys indexed:", len(meta_lookup))


In [ ]:
# 3. Enumerate all image files under DATA_ROOT and audit readability, dimensions & MD5
all_files = sorted(
    p for p in DATA_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in IMAGE_EXTS
    and "__MACOSX" not in p.parts
    and not p.name.startswith("._")
)
print(f"Total image files discovered: {len(all_files):,}")


def audit_single_image(img_path: Path):
    rel_path = norm_path(img_path.relative_to(DATA_ROOT))
    base = img_path.name
    parent_key = "/".join(rel_path.split("/")[-2:])

    label_name, generator = "unknown", "unknown"
    for key in (rel_path, parent_key, base):
        if key in meta_lookup:
            label_name, generator = meta_lookup[key]
            break
    if label_name == "unknown":
        low = f"/{rel_path.lower()}/"
        if "/real/" in low:
            label_name, generator = "real", "none"
        elif "/fake/" in low:
            label_name, generator = "fake", "unknown"

    label_int = 0 if label_name == "real" else (1 if label_name == "fake" else -1)

    try:
        raw_bytes = img_path.read_bytes()
        if len(raw_bytes) == 0:
            return {"ok": False, "rel_path": rel_path, "error": "empty_file"}
        md5_hash = hashlib.md5(raw_bytes).hexdigest()

        with Image.open(img_path) as im:
            im.verify()
        with Image.open(img_path) as im:
            width, height = im.size
            mode = im.mode
            fmt = im.format or img_path.suffix.lstrip(".").upper()

        return {
            "ok": True,
            "rel_path": rel_path,
            "label": label_int,
            "label_name": label_name,
            "generator": generator,
            "width": int(width),
            "height": int(height),
            "mode": str(mode),
            "format": str(fmt),
            "file_ext": img_path.suffix.lower(),
            "md5_hash": md5_hash,
        }
    except Exception as exc:
        return {"ok": False, "rel_path": rel_path, "error": str(exc)}


records = []
corrupt = []
with ThreadPoolExecutor(max_workers=8) as ex:
    for res in tqdm(ex.map(audit_single_image, all_files), total=len(all_files), desc="Auditing DS0001"):
        if res["ok"]:
            records.append(res)
        else:
            corrupt.append(res)

print(f"Readable images: {len(records):,} | Corrupt images: {len(corrupt):,}")


In [ ]:
# 4. Build canonical ds0001_index.csv and ds0001_audit_summary.json
df_idx = pd.DataFrame(records).sort_values("rel_path").reset_index(drop=True)
df_idx.insert(0, "image_id", [f"DS0001_{i:06d}" for i in range(1, len(df_idx) + 1)])

hash_counts = df_idx["md5_hash"].value_counts()
dup_hash_groups = int((hash_counts > 1).sum())
dup_extra_files = int((hash_counts[hash_counts > 1] - 1).sum())

audit_summary = {
    "dataset_id": "DS0001",
    "dataset_name": "ArtiFact Face Subset (50k)",
    "kaggle_slug": "isthisdeception/ds0001-artifact-face-subset",
    "license": "MIT",
    "built_utc": datetime.now(timezone.utc).isoformat(),
    "total_scanned": len(all_files),
    "readable_count": len(df_idx),
    "corrupt_count": len(corrupt),
    "corrupt_rate": round(len(corrupt) / max(len(all_files), 1), 6),
    "class_counts": df_idx["label_name"].value_counts().to_dict(),
    "generator_counts": df_idx["generator"].value_counts().to_dict(),
    "num_fake_generators": int(df_idx.loc[df_idx["label"] == 1, "generator"].nunique()),
    "width_min": int(df_idx["width"].min()),
    "width_max": int(df_idx["width"].max()),
    "height_min": int(df_idx["height"].min()),
    "height_max": int(df_idx["height"].max()),
    "format_histogram": df_idx["format"].value_counts().to_dict(),
    "mode_histogram": df_idx["mode"].value_counts().to_dict(),
    "duplicate_md5_groups": dup_hash_groups,
    "duplicate_md5_extra_files": dup_extra_files,
}

CANONICAL_COLS = [
    "image_id", "rel_path", "label", "label_name", "generator",
    "width", "height", "file_ext", "md5_hash",
]
index_csv_path = OUTPUT_DIR / "ds0001_index.csv"
summary_json_path = OUTPUT_DIR / "ds0001_audit_summary.json"

df_idx[CANONICAL_COLS].to_csv(index_csv_path, index=False)
summary_json_path.write_text(json.dumps(audit_summary, indent=2), encoding="utf-8")

print("Saved:", index_csv_path)
print("Saved:", summary_json_path)
print(json.dumps(audit_summary, indent=2))

assert len(df_idx) == 50000, f"Expected 50000 images, found {len(df_idx)}"
assert len(corrupt) == 0, f"Found {len(corrupt)} corrupt images!"
assert audit_summary["class_counts"].get("real") == 25000
assert audit_summary["class_counts"].get("fake") == 25000
assert audit_summary["num_fake_generators"] == 13
print("\n[PASS] All DS0001 integrity assertions passed!")


In [ ]:
# 5. Preview annotated examples (4 Real + 4 Diverse AI Generators) for Milestone 2 evidence
real_samples = df_idx[df_idx["label"] == 0].sample(4, random_state=SEED)
fake_gens = ["stable_diffusion", "stylegan2", "stylegan3", "taming_transformer"]
fake_samples = pd.concat([
    df_idx[df_idx["generator"] == g].sample(1, random_state=SEED)
    for g in fake_gens if (df_idx["generator"] == g).any()
])
preview_df = pd.concat([real_samples, fake_samples], ignore_index=True)

fig, axes = plt.subplots(2, 4, figsize=(12, 6.5), dpi=200)
for ax, row in zip(axes.flat, preview_df.itertuples(index=False)):
    im = Image.open(DATA_ROOT / row.rel_path)
    ax.imshow(im)
    color = "#1b7f3b" if row.label == 0 else "#b42318"
    ax.set_title(
        f"{row.label_name.upper()} | {row.generator}\n{row.image_id} ({row.width}x{row.height})",
        fontsize=8.5,
        color=color,
        fontweight="bold",
    )
    ax.axis("off")

plt.suptitle("DS0001 (ArtiFact Face Subset) — Verified Annotated Sample Preview", fontsize=12, fontweight="bold")
plt.tight_layout()
preview_path = OUTPUT_DIR / "ds0001_annotation_examples.png"
plt.savefig(preview_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved annotation preview:", preview_path)
preview_df[CANONICAL_COLS[:7]]
